# Windowing + PCA — Extended Features (CC1 + CC2 only)

**New notebook.** Input: `data/processed_extended/{cc1_train,cc1_val,cc1_test,drift_cc2}.csv`
(from `clean_and_split_extended.ipynb`). Same window size (30, unchanged
deliberately — isolating the feature-engineering variable, not conflating it
with a window-size change) applied to the **11-feature** set (330-dim flattened,
up from 210).

**PCA variance threshold is re-examined from scratch here, not assumed.** The
original pipeline found 95% variance dropped to just 5 components and nearly
eliminated the CPU-fault signal, which is why 99%+whitening was chosen instead.
With new features added (especially the throttling rates, which are zero-heavy
like the original CPU rates), that analysis needs to be redone — it may or may
not land on the same threshold.

In [1]:
import pandas as pd
import numpy as np
import joblib, os
from sklearn.decomposition import PCA

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'experiments', 'models_extended', 'data', 'processed_extended')
OUT_DIR   = os.path.join(BASE, 'experiments', 'models_extended', 'data', 'processed_extended', 'windows_cc1_extended')
MODEL_DIR = os.path.join(BASE, 'experiments', 'models_extended', 'model')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_cpu_cfs_throttled_seconds_rate', 'container_cpu_cfs_throttled_periods_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes',
    'container_memory_rss', 'container_memory_cache', 'container_threads', 'memory_limit_proximity',
]
SPLIT_FILES = {'cc1_train': 'cc1_train.csv', 'cc1_val': 'cc1_val.csv', 'cc1_test': 'cc1_test.csv', 'drift_cc2': 'drift_cc2.csv'}

WINDOW_SIZE = 30   # unchanged from the original pipeline — isolates the feature-engineering variable
STRIDE = 1

print(f'{len(FEATURE_COLS)} features:', FEATURE_COLS)

11 features: ['container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate', 'container_cpu_cfs_throttled_seconds_rate', 'container_cpu_cfs_throttled_periods_rate', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache', 'container_threads', 'memory_limit_proximity']


## Step 1 — Load splits

In [2]:
splits = {}
for name, fname in SPLIT_FILES.items():
    d = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    splits[name] = d
    print(f'  {name:10s}: {len(d):>7,} rows  |  anomalies: {int(d["label"].sum()):,}')

  cc1_train : 156,479 rows  |  anomalies: 0
  cc1_val   :  22,356 rows  |  anomalies: 0
  cc1_test  :  44,968 rows  |  anomalies: 256
  drift_cc2 :  77,760 rows  |  anomalies: 372


## Step 2 — Sliding windows (gap-aware, same logic as the original pipeline)

In [3]:
def build_windows(container_df, feature_cols, window_size, stride):
    data = container_df[feature_cols].values.astype(np.float32)
    labels = container_df['label'].values
    ftypes = container_df['failure_type'].values.astype(object)
    is_gap = container_df['is_gap'].values
    n = len(data)
    X, y, ft = [], [], []
    for i in range(0, n - window_size + 1, stride):
        if is_gap[i:i + window_size].any():
            continue
        X.append(data[i:i + window_size])
        y.append(int(labels[i:i + window_size].any()))
        w_types = sorted({t for t in ftypes[i:i + window_size] if isinstance(t, str)})
        ft.append(','.join(w_types) if w_types else None)
    if not X:
        return (np.empty((0, window_size, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.int64), np.array([], dtype=object))
    return np.stack(X), np.array(y, dtype=np.int64), np.array(ft, dtype=object)

def window_split(df, feature_cols, window_size, stride):
    Xs, ys, fts = [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        X, y, ft = build_windows(g, feature_cols, window_size, stride)
        if len(X):
            Xs.append(X); ys.append(y); fts.append(ft)
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(fts)

windowed = {}
for name, d in splits.items():
    X, y, ft = window_split(d, FEATURE_COLS, WINDOW_SIZE, STRIDE)
    windowed[name] = {'X': X, 'y': y, 'ft': ft}
    print(f'  {name:10s}: {len(d):>7,} rows -> {len(y):>7,} windows  ({int(y.sum()):,} anomaly windows)')

for name in windowed:
    X = windowed[name]['X']
    windowed[name]['X_flat'] = X.reshape(len(X), -1)
    print(f'  {name:10s}: {X.shape} -> {windowed[name]["X_flat"].shape}')

  cc1_train : 156,479 rows -> 154,198 windows  (0 anomaly windows)
  cc1_val   :  22,356 rows ->  21,573 windows  (0 anomaly windows)
  cc1_test  :  44,968 rows ->  44,185 windows  (256 anomaly windows)
  drift_cc2 :  77,760 rows ->  76,977 windows  (720 anomaly windows)
  cc1_train : (154198, 30, 11) -> (154198, 330)
  cc1_val   : (21573, 30, 11) -> (21573, 330)
  cc1_test  : (44185, 30, 11) -> (44185, 330)
  drift_cc2 : (76977, 30, 11) -> (76977, 330)


## Step 3 — Re-examine the PCA variance threshold (don't assume 99% still applies)

Check both 95% and 99% and inspect which raw features the borderline
components load onto, exactly as done for the original 7-feature set.

In [4]:
X_train_flat = windowed['cc1_train']['X_flat']
print(f'Input dimension: {X_train_flat.shape[1]} (= {WINDOW_SIZE} timesteps x {len(FEATURE_COLS)} features)')

for variance_target in [0.95, 0.99]:
    pca_check = PCA(n_components=variance_target, svd_solver='full', random_state=42)
    pca_check.fit(X_train_flat)
    n_comp = pca_check.n_components_
    print(f'\n{variance_target*100:.0f}% variance -> {n_comp} components')
    print(f'  explained_variance_ratio_ (first 10): {np.round(pca_check.explained_variance_ratio_[:10], 4)}')
    comps = pca_check.components_.reshape(n_comp, WINDOW_SIZE, len(FEATURE_COLS))
    for i in range(min(n_comp, 10)):
        loading = np.abs(comps[i]).sum(axis=0)
        top_feat = FEATURE_COLS[np.argmax(loading)]
        throttle_share = (loading[3] + loading[4]) / loading.sum()   # throttled-rate features' share of this component's loading
        thread_share = loading[9] / loading.sum()
        print(f'    PC{i+1}: var={pca_check.explained_variance_ratio_[i]:.4f}  dominant={top_feat:42s}  '
              f'throttle_share={throttle_share:.3f}  thread_share={thread_share:.3f}')

Input dimension: 330 (= 30 timesteps x 11 features)

95% variance -> 3 components
  explained_variance_ratio_ (first 10): [0.7297 0.1568 0.073 ]
    PC1: var=0.7297  dominant=container_threads                           throttle_share=0.000  thread_share=0.330
    PC2: var=0.1568  dominant=container_threads                           throttle_share=0.000  thread_share=0.284
    PC3: var=0.0730  dominant=container_memory_cache                      throttle_share=0.000  thread_share=0.060

99% variance -> 26 components
  explained_variance_ratio_ (first 10): [0.7297 0.1568 0.073  0.0041 0.0041 0.0033 0.0021 0.0014 0.0014 0.0012]
    PC1: var=0.7297  dominant=container_threads                           throttle_share=0.000  thread_share=0.330
    PC2: var=0.1568  dominant=container_threads                           throttle_share=0.000  thread_share=0.284
    PC3: var=0.0730  dominant=container_memory_cache                      throttle_share=0.000  thread_share=0.060
    PC4: var=0.0041  d

## Step 4 — Fit PCA at the chosen threshold, transform every split

Uses 99% + whitening by default, matching the original pipeline's justification
(preserve low-variance-but-informative components) — the printed breakdown
above is there to confirm or correct that choice for the extended feature set,
not to blindly repeat it.

In [5]:
PCA_VARIANCE = 0.99
pca = PCA(n_components=PCA_VARIANCE, svd_solver='full', whiten=True, random_state=42)
pca.fit(X_train_flat)
n_components = pca.n_components_
print(f'Components retained for {PCA_VARIANCE*100:.0f}% variance: {n_components}  (compression {X_train_flat.shape[1]} -> {n_components})')

for name in windowed:
    windowed[name]['X_pca'] = pca.transform(windowed[name]['X_flat']).astype(np.float32)
    print(f'  {name:10s}: {windowed[name]["X_flat"].shape} -> {windowed[name]["X_pca"].shape}')

print()
print('Whitened cc1_train PCA-space stats (all components should be ~unit variance):')
stats_df = pd.DataFrame(windowed['cc1_train']['X_pca']).describe().loc[['mean', 'std']]
print(f'  mean range: [{stats_df.loc["mean"].min():.4f}, {stats_df.loc["mean"].max():.4f}]')
print(f'  std  range: [{stats_df.loc["std"].min():.4f}, {stats_df.loc["std"].max():.4f}]')

Components retained for 99% variance: 26  (compression 330 -> 26)
  cc1_train : (154198, 330) -> (154198, 26)
  cc1_val   : (21573, 330) -> (21573, 26)
  cc1_test  : (44185, 330) -> (44185, 26)
  drift_cc2 : (76977, 330) -> (76977, 26)

Whitened cc1_train PCA-space stats (all components should be ~unit variance):
  mean range: [-0.0002, 0.0001]
  std  range: [1.0000, 1.0000]


## Step 5 — Save

In [6]:
for name, d in windowed.items():
    np.save(os.path.join(OUT_DIR, f'X_{name}.npy'), d['X_pca'])
    np.save(os.path.join(OUT_DIR, f'y_{name}.npy'), d['y'])
    np.save(os.path.join(OUT_DIR, f'ft_{name}.npy'), d['ft'])
    print(f'  saved X/y/ft_{name}.npy  ({len(d["y"]):,} windows)')

joblib.dump({'pca': pca, 'feature_cols': FEATURE_COLS, 'window_size': WINDOW_SIZE, 'stride': STRIDE, 'n_components': n_components},
            os.path.join(MODEL_DIR, 'cc1_pca_extended.pkl'))
print(f'\nPCA model saved. n_components={n_components} (original 7-feature pipeline used 26).')

  saved X/y/ft_cc1_train.npy  (154,198 windows)
  saved X/y/ft_cc1_val.npy  (21,573 windows)
  saved X/y/ft_cc1_test.npy  (44,185 windows)
  saved X/y/ft_drift_cc2.npy  (76,977 windows)

PCA model saved. n_components=26 (original 7-feature pipeline used 26).


## Summary

| Output | Shape |
|---|---|
| `data/processed_extended/windows_cc1_extended/X_*.npy` | (N, n_components) |
| `models_extended/cc1_pca_extended.pkl` | fitted PCA + config |

Next: `train_vae_extended.ipynb` — trains on this extended representation,
re-running the beta/latent-dim ablations since the input dimension changed.